<a href="https://colab.research.google.com/github/fisherat/CSFinalProjectSp26/blob/main/fishera_FD_CSFinalPrj_Webscraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Import relevant libraries - Webscraping
import requests
from bs4 import BeautifulSoup

#Import relevant libraries - Data cleaning & Preparations for export
import pandas as pd
import re

#Import failure libraries for workflow sanity-checks.
from warnings import filterwarnings

#Save the ChemLibretext urls for Webscraping
gc_url = 'https://chem.libretexts.org/Bookshelves/General_Chemistry/Chemistry_2e_(OpenStax)'
oc_url = 'https://chem.libretexts.org/Bookshelves/Organic_Chemistry/Organic_Chemistry_(OpenStax)'
ic_url = 'https://chem.libretexts.org/Bookshelves/Inorganic_Chemistry/Inorganic_Chemistry_(LibreTexts)'
ac_url = 'https://chem.libretexts.org/Bookshelves/Analytical_Chemistry/Instrumental_Analysis_(LibreTexts)'
pc_url = 'https://chem.libretexts.org/Bookshelves/Physical_and_Theoretical_Chemistry_Textbook_Maps/Physical_Chemistry_(LibreTexts)'

#Biochemistry and Environmental chemistry don't have a Libretext/OpenStax textbook, so we should handle them differently
bc_url = 'https://chem.libretexts.org/Bookshelves/Biological_Chemistry'
ec_url = 'https://chem.libretexts.org/Bookshelves/Environmental_Chemistry'

#Webscrape and save the url as an HTML using requests.get()
def get_chapter_data(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    chapters = soup.find_all('a', class_='mt-sortable-listing-link mt-edit-section internal')
    chapter_data = []
    for chapter in chapters:
        title_tag = chapter.find('span', class_='mt-sortable-listing-title')
        title = title_tag.get_text() if title_tag else 'No Title'
        link = chapter.get('href')
        chapter_data.append({'title': title, 'link': link})
    return chapter_data

gc_chapter_data = pd.DataFrame(get_chapter_data(gc_url))
oc_chapter_data = pd.DataFrame(get_chapter_data(oc_url))
ic_chapter_data = pd.DataFrame(get_chapter_data(ic_url))
ac_chapter_data = pd.DataFrame(get_chapter_data(ac_url))
pc_chapter_data = pd.DataFrame(get_chapter_data(pc_url))

#bc_chapter_data = pd.DataFrame(get_chapter_data(bc_url))
#ec_chapter_data = pd.DataFrame(get_chapter_data(ec_url))

def get_unit_count(df):
    for index, row in df.iterrows():
        chapter_url = row["link"]
        chapter_html = requests.get(chapter_url)
        chapter_soup = BeautifulSoup(chapter_html.content, "html.parser")
        units = chapter_soup.find_all("li", class_ = 'mt-list-topics')
        df.at[index, '# Units'] = len(units)
    return df

gc_chapters_df = get_unit_count(gc_chapter_data)
oc_chapters_df = get_unit_count(oc_chapter_data)
ic_chapters_df = get_unit_count(ic_chapter_data)
ac_chapters_df = get_unit_count(ac_chapter_data)
pc_chapters_df = get_unit_count(pc_chapter_data)

#bc_chapters_df = get_unit_count(bc_chapter_data)
#ec_chapters_df = get_unit_count(ec_chapter_data)


In [ ]:
#Collect Unit Data in a seperate df so as not to collect word counts for chapter introductions
def units_per_chapter(df):
  all_unit_data = []
  for index, row in df.iterrows():
    chapter_title = row['title'] # Define chapter_title from the current chapter row
    chapter_url = row['link']   # Define chapter_url from the current chapter row

    unit_html = requests.get(row['link'])
    unit_soup = BeautifulSoup(unit_html.content, "html.parser")
    units = unit_soup.find_all("dl", class_ = 'mt-listing-detailed')
    for unit_dl_element in units:
        dt_tag = unit_dl_element.find('dt')
        if dt_tag:
            unit_link_tag = dt_tag.find('a', class_='mt-sortable-listing-link')
            if not unit_link_tag:
                    # If not found with class, try finding any <a> tag within <dt>
                    unit_link_tag = dt_tag.find('a')

            if unit_link_tag:
                    # Now extract title and link from unit_link_tag
                    unit_title_tag = unit_link_tag.find('span', class_='mt-sortable-listing-title')
                    if unit_title_tag:
                        unit_title = unit_title_tag.get_text()
                    else:
                        # Fallback: try to get text directly from the <a> tag
                        unit_title = unit_link_tag.get_text(strip=True) if unit_link_tag.get_text(strip=True) else 'No Unit Title'
                    unit_link = unit_link_tag.get('href')

                    all_unit_data.append({'chapter_title': chapter_title,'unit_title': unit_title, 'unit_link': unit_link})
            else:
                    print(f"Warning: Unit in chapter '{chapter_title}' (inside <dt>) at {chapter_url} did not have a valid link (<a> tag not found).")
        else:
                print(f"Warning: Unit in chapter '{chapter_title}' at {chapter_url} did not have a <dt> tag inside <dl>.")
  return pd.DataFrame(all_unit_data) # Return a new DataFrame with unit data

fullgcdf = units_per_chapter(gc_chapters_df)
fullgcdf['specialty'] = 'General'
fullocdf = units_per_chapter(oc_chapters_df)
fullocdf['specialty'] = 'Organic'
fullicdf = units_per_chapter(ic_chapters_df)
fullicdf['specialty'] = 'Inorganic'
fullacdf = units_per_chapter(ac_chapters_df)
fullacdf['specialty'] = 'Analytical & Instrumental'
fullpcdf = units_per_chapter(pc_chapters_df)
fullpcdf['specialty'] = 'Physical & Theoretical'

#fullbcdf = units_per_chapter(bc_chapters_df)
#fullecdf = units_per_chapter(ec_chapters_df)


In [ ]:
#The less you have to run the block above, the better.

#Clean out some of the useless pages (Titles, Info, T.O.C., Licensing, etc.)
# List of unit titles to exclude
titles_to_exclude = ['TitlePage', 'InfoPage', 'Table of Contents', 'Licensing', 'Detailed Licensing',
                     'Back Matter', 'Front Matter', 'Appendices', 'Appendix','Index',
                     'Glossary', 'Answer Keys', 'Problems']

# Function to filter dataframes based on unit_title
def clean_titles(df):
  # Check if DataFrame is empty or missing required columns
  if df.empty or 'unit_title' not in df.columns or 'chapter_title' not in df.columns:
    return df

  # Create a combined regex pattern from titles_to_exclude
  # Escape special characters in regex to treat them literally (e.g., '.')
  pattern = '|'.join([re.escape(word.strip()) for word in titles_to_exclude])

  # Use vectorized string operations with regex to identify rows to remove
  # Match if any word from the exclude list is found in unit_title or chapter_title (case-insensitive)
  mask_unit_title = df['unit_title'].astype(str).str.contains(pattern, case=False, na=False)
  mask_chapter_title = df['chapter_title'].astype(str).str.contains(pattern, case=False, na=False)

  # Combine masks: mark a row for removal if it matches in either unit_title or chapter_title
  mask_to_remove = mask_unit_title | mask_chapter_title

  # Invert the mask to get rows to keep
  return df[~mask_to_remove]

fullgcdf = clean_titles(fullgcdf)
fullocdf = clean_titles(fullocdf)
fullicdf = clean_titles(fullicdf)
fullacdf = clean_titles(fullacdf)
fullpcdf = clean_titles(fullpcdf)

In [ ]:
#Sanity check of cleaned dataframes
display(fullgcdf)
display(fullocdf)
display(fullicdf)
display(fullacdf)
display(fullpcdf)

#display(fullbcdf)
#display(fullecdf)

,chapter_title,unit_title,unit_link,specialty
4,1: Essential Ideas,1.0: Introduction,https://chem.libretexts.org/Bookshelves/Genera...,General
5,1: Essential Ideas,1.1: Chemistry in Context,https://chem.libretexts.org/Bookshelves/Genera...,General
6,1: Essential Ideas,1.2: Phases and Classification of Matter,https://chem.libretexts.org/Bookshelves/Genera...,General
7,1: Essential Ideas,1.3: Physical and Chemical Properties,https://chem.libretexts.org/Bookshelves/Genera...,General
8,1: Essential Ideas,1.4: Measurements,https://chem.libretexts.org/Bookshelves/Genera...,General
...,...,...,...,...
215,21: Nuclear Chemistry,21.6: Biological Effects of Radiation,https://chem.libretexts.org/Bookshelves/Genera...,General
216,21: Nuclear Chemistry,21.7: Key Terms,https://chem.libretexts.org/Bookshelves/Genera...,General
217,21: Nuclear Chemistry,21.8: Key Equations,https://chem.libretexts.org/Bookshelves/Genera...,General
218,21: Nuclear Chemistry,21.9: Summary,https://chem.libretexts.org/Bookshelves/Genera...,General


,chapter_title,unit_title,unit_link,specialty
4,1: Structure and Bonding,1.0: Why This Chapter?,https://chem.libretexts.org/Bookshelves/Organi...,Organic
5,1: Structure and Bonding,1.1: Atomic Structure - The Nucleus,https://chem.libretexts.org/Bookshelves/Organi...,Organic
6,1: Structure and Bonding,1.2: Atomic Structure - Orbitals,https://chem.libretexts.org/Bookshelves/Organi...,Organic
7,1: Structure and Bonding,1.3: Atomic Structure - Electron Configurations,https://chem.libretexts.org/Bookshelves/Organi...,Organic
8,1: Structure and Bonding,1.4: Development of Chemical Bonding Theory,https://chem.libretexts.org/Bookshelves/Organi...,Organic
...,...,...,...,...
476,31: Synthetic Polymers,31.6: Intramolecular Olefin Metathesis,https://chem.libretexts.org/Bookshelves/Organi...,Organic
477,31: Synthetic Polymers,31.7: Polymer Structure and Physical Properties,https://chem.libretexts.org/Bookshelves/Organi...,Organic
478,31: Synthetic Polymers,31.8: Chemistry Matters—Degradable Polymers,https://chem.libretexts.org/Bookshelves/Organi...,Organic
479,31: Synthetic Polymers,31.9: Key Terms,https://chem.libretexts.org/Bookshelves/Organi...,Organic


,chapter_title,unit_title,unit_link,specialty
4,1: Introduction to Inorganic Chemistry,1.1: What is Inorganic Chemistry?,https://chem.libretexts.org/Bookshelves/Inorga...,Inorganic
5,1: Introduction to Inorganic Chemistry,1.2: Inorganic vs Organic Chemistry,https://chem.libretexts.org/Bookshelves/Inorga...,Inorganic
6,1: Introduction to Inorganic Chemistry,1.3: History of Inorganic Chemistry,https://chem.libretexts.org/Bookshelves/Inorga...,Inorganic
7,1: Introduction to Inorganic Chemistry,1.4: Perspectives,https://chem.libretexts.org/Bookshelves/Inorga...,Inorganic
9,2: Atomic Structure,2.1: Historical Development of Atomic Theory,https://chem.libretexts.org/Bookshelves/Inorga...,Inorganic
...,...,...,...,...
93,14: Organometallic Reactions and Catalysis,14.4: Heterogeneous Catalysts,https://chem.libretexts.org/Bookshelves/Inorga...,Inorganic
95,15: Parallels between Main Group and Organomet...,15.1: Parallels between Main Group and Binary ...,https://chem.libretexts.org/Bookshelves/Inorga...,Inorganic
96,15: Parallels between Main Group and Organomet...,15.2: The Isolobal Analogy,https://chem.libretexts.org/Bookshelves/Inorga...,Inorganic
97,15: Parallels between Main Group and Organomet...,15.3: Metal-Metal Bonds,https://chem.libretexts.org/Bookshelves/Inorga...,Inorganic


,chapter_title,unit_title,unit_link,specialty
5,1: Introduction,1.1: Classification of Analytical Methods,https://chem.libretexts.org/Bookshelves/Analyt...,Analytical & Instrumental
6,1: Introduction,1.2: Types of Instrumental Methods,https://chem.libretexts.org/Bookshelves/Analyt...,Analytical & Instrumental
7,1: Introduction,1.3: Instruments For Analysis,https://chem.libretexts.org/Bookshelves/Analyt...,Analytical & Instrumental
8,1: Introduction,1.4: Selecting an Analytical Method,https://chem.libretexts.org/Bookshelves/Analyt...,Analytical & Instrumental
9,1: Introduction,1.5: Calibration of Instrumental Methods,https://chem.libretexts.org/Bookshelves/Analyt...,Analytical & Instrumental
...,...,...,...,...
154,35: Appendicies,35.6: Critical Values for Grubb's Test,https://chem.libretexts.org/Bookshelves/Analyt...,Analytical & Instrumental
155,35: Appendicies,35.7: Activity Coefficients,https://chem.libretexts.org/Bookshelves/Analyt...,Analytical & Instrumental
156,35: Appendicies,35.8: Standard Reduction Potentials & Polarogr...,https://chem.libretexts.org/Bookshelves/Analyt...,Analytical & Instrumental
157,35: Appendicies,35.9: Recommended Primary Standards,https://chem.libretexts.org/Bookshelves/Analyt...,Analytical & Instrumental


,chapter_title,unit_title,unit_link,specialty
4,1: The Dawn of the Quantum Theory,1.1: Blackbody Radiation Cannot Be Explained C...,https://chem.libretexts.org/Bookshelves/Physic...,Physical & Theoretical
5,1: The Dawn of the Quantum Theory,1.2: Quantum Hypothesis Used for Blackbody Rad...,https://chem.libretexts.org/Bookshelves/Physic...,Physical & Theoretical
6,1: The Dawn of the Quantum Theory,1.3: Photoelectric Effect Explained with Quant...,https://chem.libretexts.org/Bookshelves/Physic...,Physical & Theoretical
7,1: The Dawn of the Quantum Theory,1.4: The Hydrogen Atomic Spectrum,https://chem.libretexts.org/Bookshelves/Physic...,Physical & Theoretical
8,1: The Dawn of the Quantum Theory,1.5: The Rydberg Formula and the Hydrogen Atom...,https://chem.libretexts.org/Bookshelves/Physic...,Physical & Theoretical
...,...,...,...,...
331,32: Math Chapters,32.7: Determinants,https://chem.libretexts.org/Bookshelves/Physic...,Physical & Theoretical
332,32: Math Chapters,32.8: Matrices,https://chem.libretexts.org/Bookshelves/Physic...,Physical & Theoretical
333,32: Math Chapters,32.9: Series and Limits,https://chem.libretexts.org/Bookshelves/Physic...,Physical & Theoretical
334,32: Math Chapters,32.10: Fourier Analysis,https://chem.libretexts.org/Bookshelves/Physic...,Physical & Theoretical


In [ ]:
#Measure relative difficulty of content by ACS exam statistics
#Latest available Statistics published by:
#ACS Division of Chemical Education Examinations Institute & University of Wisconsin - Milwaukee
#National Norms. ACS Exams. https://uwm.edu/acs-exams/instructors/exam-statistics/national-norms/
meanscore = {'Genchem' : 37.8/70, 'Orgo' : 36.7/70, 'Analytic' : 29.1/50, 'Phys' : 35.4/60,'Inorganic' : 34.4/60}
medianscore ={'Genchem' : 37/70, 'Orgo' : 35/70, 'Analytic': 29/50, 'Phys' : 36/60, 'Inorganic' : 34/60}
fullgcdf['mean_score'] = meanscore['Genchem']
fullgcdf['median_score'] = medianscore['Genchem']
fullocdf['mean_score'] = meanscore['Orgo']
fullocdf['median_score'] = medianscore['Orgo']
fullacdf['mean_score'] = meanscore['Analytic']
fullacdf['median_score'] = medianscore['Analytic']
fullpcdf['mean_score'] = meanscore['Phys']
fullpcdf['median_score'] = medianscore['Phys']
fullicdf['mean_score'] = meanscore['Inorganic']
fullicdf['median_score'] = medianscore['Inorganic']

In [ ]:
#Put all of the dataframes together
libretexts = pd.concat([fullgcdf, fullocdf, fullacdf, fullpcdf, fullicdf], ignore_index=True)
display(libretexts)

,chapter_title,unit_title,unit_link,specialty,mean_score,median_score
0,1: Essential Ideas,1.0: Introduction,https://chem.libretexts.org/Bookshelves/Genera...,General,0.540000,0.528571
1,1: Essential Ideas,1.1: Chemistry in Context,https://chem.libretexts.org/Bookshelves/Genera...,General,0.540000,0.528571
2,1: Essential Ideas,1.2: Phases and Classification of Matter,https://chem.libretexts.org/Bookshelves/Genera...,General,0.540000,0.528571
3,1: Essential Ideas,1.3: Physical and Chemical Properties,https://chem.libretexts.org/Bookshelves/Genera...,General,0.540000,0.528571
4,1: Essential Ideas,1.4: Measurements,https://chem.libretexts.org/Bookshelves/Genera...,General,0.540000,0.528571
...,...,...,...,...,...,...
1226,14: Organometallic Reactions and Catalysis,14.4: Heterogeneous Catalysts,https://chem.libretexts.org/Bookshelves/Inorga...,Inorganic,0.573333,0.566667
1227,15: Parallels between Main Group and Organomet...,15.1: Parallels between Main Group and Binary ...,https://chem.libretexts.org/Bookshelves/Inorga...,Inorganic,0.573333,0.566667
1228,15: Parallels between Main Group and Organomet...,15.2: The Isolobal Analogy,https://chem.libretexts.org/Bookshelves/Inorga...,Inorganic,0.573333,0.566667
1229,15: Parallels between Main Group and Organomet...,15.3: Metal-Metal Bonds,https://chem.libretexts.org/Bookshelves/Inorga...,Inorganic,0.573333,0.566667


In [ ]:
#Webscrape further for more data, now on each individual unit within the bookshelves (details on unit 1.1 rather than just the title and link)
def get_content_metrics(unit_url):
    try:
        response = requests.get(unit_url, timeout=10)
        response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)
        soup = BeautifulSoup(response.content, 'html.parser')

        content_div = None
        # Try LibreTexts specific ID first
        content_div = soup.find('div', id='mt-content-container')
        # Then try generic article tag
        if not content_div:
            content_div = soup.find('article')
        # Then try common content div classes
        if not content_div:
            content_div = soup.find('div', class_='main-content')
        if not content_div:
            content_div = soup.find('div', class_='body-content')

        word_count = 0
        # Extract text specifically from <p> tags
        if content_div:
            paragraphs = content_div.find_all('p')
        else:
            # If no specific content container is found, look for <p> tags in the whole soup
            paragraphs = soup.find_all('p')
            if not paragraphs:
                print(f"Warning: No specific content container or <p> tags found for {unit_url}. Word count set to 0.")

        if paragraphs:
            text_content = ' '.join([p.get_text(separator=' ', strip=True) for p in paragraphs])
            word_count = len([word for word in text_content.split() if word])

        # Figure count can still be from the whole soup if no specific content_div is found, as figures are often outside the strict text content but still relevant.
        target_scope_for_figures = content_div if content_div else soup
        img_count = len(target_scope_for_figures.find_all('img'))
        svg_count = len(target_scope_for_figures.find_all('svg'))
        figure_count = img_count + svg_count

        return word_count, figure_count
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {unit_url}: {e}")
        return 0, 0 # Return 0 for counts in case of error


In [ ]:
# Re-apply the content metrics function to each unit link with the updated definition
print("Calculating word and figure counts for each unit, this may take a few minutes")
libretexts[['word_count', 'figure_count']] = libretexts['unit_link'].apply(lambda x: pd.Series(get_content_metrics(x)))

print("Content metric recalculation complete.")
# Re-display head and info to show updated columns
display(libretexts.head())
libretexts.info()


Calculating word and figure counts for each unit, this may take a few minutes
Content metric recalculation complete.


,chapter_title,unit_title,unit_link,specialty,mean_score,median_score,word_count,figure_count
0,1: Essential Ideas,1.0: Introduction,https://chem.libretexts.org/Bookshelves/Genera...,General,0.54,0.528571,339,1
1,1: Essential Ideas,1.1: Chemistry in Context,https://chem.libretexts.org/Bookshelves/Genera...,General,0.54,0.528571,1741,5
2,1: Essential Ideas,1.2: Phases and Classification of Matter,https://chem.libretexts.org/Bookshelves/Genera...,General,0.54,0.528571,2871,13
3,1: Essential Ideas,1.3: Physical and Chemical Properties,https://chem.libretexts.org/Bookshelves/Genera...,General,0.54,0.528571,1012,6
4,1: Essential Ideas,1.4: Measurements,https://chem.libretexts.org/Bookshelves/Genera...,General,0.54,0.528571,2099,4


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1231 entries, 0 to 1230
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   chapter_title  1231 non-null   object 
 1   unit_title     1231 non-null   object 
 2   unit_link      1231 non-null   object 
 3   specialty      1231 non-null   object 
 4   mean_score     1231 non-null   float64
 5   median_score   1231 non-null   float64
 6   word_count     1231 non-null   int64  
 7   figure_count   1231 non-null   int64  
dtypes: float64(2), int64(2), object(4)
memory usage: 77.1+ KB


In [ ]:
#Webscraping and initial sanity checking is complete, now lets port it and put it into another code
#Download and save the .csv on your PC
libretexts.to_csv('libretexts_data.csv', index=False)
libretexts.head()

,chapter_title,unit_title,unit_link,specialty,mean_score,median_score,word_count,figure_count
0,1: Essential Ideas,1.0: Introduction,https://chem.libretexts.org/Bookshelves/Genera...,General,0.54,0.528571,339,1
1,1: Essential Ideas,1.1: Chemistry in Context,https://chem.libretexts.org/Bookshelves/Genera...,General,0.54,0.528571,1741,5
2,1: Essential Ideas,1.2: Phases and Classification of Matter,https://chem.libretexts.org/Bookshelves/Genera...,General,0.54,0.528571,2871,13
3,1: Essential Ideas,1.3: Physical and Chemical Properties,https://chem.libretexts.org/Bookshelves/Genera...,General,0.54,0.528571,1012,6
4,1: Essential Ideas,1.4: Measurements,https://chem.libretexts.org/Bookshelves/Genera...,General,0.54,0.528571,2099,4
